# Mage ML pipeline: assignment version (Part C)
Based on the class notebook **Maze_ML_PipeLine**. The Mage project `iris_project/` is already in this repository,
so this notebook clones the repo, installs Mage, and runs C1–C4. Run top to bottom in Google Colab.

## 0. Get the repository and install Mage

In [ ]:
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/freshcart-pipelines.git"   # <- your repo
!git clone -q {REPO_URL} /content/freshcart-pipelines
%cd /content/freshcart-pipelines
!ls

In [ ]:
!pip install -q uv
!uv venv -q -p 3.12 /content/mage_env
!uv pip install -q --python /content/mage_env mage-ai scikit-learn

import os
os.environ['PATH'] = '/content/mage_env/bin:' + os.environ['PATH']
!mage --help | head -5

## C1. The pipeline and a run with tests
Final graph (after C3 and C4): `load_iris → prepare_data → {train_model → {save_results, evaluate_model}, train_knn}`.

In [ ]:
!cat iris_project/pipelines/iris_classifier/metadata.yaml

In [ ]:
!mage run iris_project iris_classifier --test 2>&1 | grep -v traitlets

## C2. Tests as a gate
For each max_depth: delete `iris_predictions.csv`, run with `--runtime-vars`, record accuracy, whether
`test_accuracy` passed, and whether the CSV was created.

In [ ]:
import subprocess, re, json, pandas as pd

rows = []
for depth in [1, 2, 3, 4, 5]:
    if os.path.exists('iris_predictions.csv'):
        os.remove('iris_predictions.csv')
    out = subprocess.run(['mage', 'run', 'iris_project', 'iris_classifier', '--test',
                          '--runtime-vars', json.dumps({'max_depth': depth})],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout
    acc = re.search(r'max_depth = \d+ \| Test accuracy = ([0-9.]+)', out).group(1)
    print(f'----- max_depth = {depth} -----')
    print('\n'.join(l for l in out.splitlines() if 'accuracy' in l or 'FAIL' in l or 'Saved' in l or 'tests passed' in l))
    rows.append({'max_depth': depth, 'test_accuracy': float(acc),
                 'test_accuracy passed?': 'no' if 'FAIL: test_accuracy' in out else 'yes',
                 'iris_predictions.csv created?': 'yes' if os.path.exists('iris_predictions.csv') else 'no'})

pd.DataFrame(rows)

Full log of the failing run (max_depth = 1), to show the test stopping the pipeline:

In [ ]:
!rm -f iris_predictions.csv
!mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 1}' 2>&1 | grep -v traitlets | head -25
print('iris_predictions.csv exists:', os.path.exists('iris_predictions.csv'))

## C3. Evaluation branch: confusion matrix and metrics (max_depth = 3)

In [ ]:
!cat iris_project/transformers/evaluate_model.py

In [ ]:
!mage run iris_project iris_classifier --test 2>&1 | grep -v traitlets

Same branch with max_depth = 2, where the matrix is more interesting (one mistake):

In [ ]:
!mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 2}' 2>&1 | sed -n '/Confusion/,/f1_score/p'

## C4. Own block: a KNN classifier as a parallel branch with its own @test

In [ ]:
!cat iris_project/transformers/train_knn.py

In [ ]:
!mage run iris_project iris_classifier --test 2>&1 | grep -E 'accuracy|tests passed|completed'

CI check, the same command GitHub Actions runs:

In [ ]:
!/content/mage_env/bin/python -m pip install -q pytest && /content/mage_env/bin/python -m pytest -v

## Mage UI (screenshot for C3): start the server and open it through a Cloudflare tunnel

In [ ]:
import subprocess, time, urllib.request
mage_server = subprocess.Popen(['/content/mage_env/bin/mage', 'start', 'iris_project'],
                               stdout=open('mage.log', 'w'), stderr=subprocess.STDOUT)
for i in range(60):
    try:
        urllib.request.urlopen('http://localhost:6789'); print('Mage is running'); break
    except Exception:
        time.sleep(2)
else:
    print('Mage did not start - check mage.log')

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:6789'],
                          stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(10)
link = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
print('Open Mage here:', link[0] if link else 'Not ready yet - run this cell again')
print('Then: Pipelines -> iris_classifier -> Edit, and screenshot the graph showing the branch.')